# Masking version 이미지 생성 파일

TS/TL 조합의 1, 3~8번 zip을 가져와 74종 외 알약은 마스킹한 이미지를 생성합니다.

In [ ]:
import os, re, json, io, zipfile, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from collections import defaultdict
from google.colab import drive

drive.mount('/content/drive')

# ── 경로 설정 ──────────────────────────────────────────────
TEAM_DIR   = "/content/drive/MyDrive"
DRIVE_BASE = "/content/drive/MyDrive"
OUTPUT_IMG_56 = os.path.join(DRIVE_BASE, "dataset_56_masked", "images")
OUTPUT_ANN_56 = os.path.join(DRIVE_BASE, "dataset_56_masked", "annotations")
OUTPUT_IMG_74 = os.path.join(DRIVE_BASE, "dataset_74_masked", "images")
OUTPUT_ANN_74 = os.path.join(DRIVE_BASE, "dataset_74_masked", "annotations")
for d in [OUTPUT_IMG_56, OUTPUT_ANN_56, OUTPUT_IMG_74, OUTPUT_ANN_74]:
    os.makedirs(d, exist_ok=True)

ZIP_NUMS     = [1, 3, 4, 5, 6, 7, 8]
MASK_PADDING = 10
CENTER_SIZE  = 100

# ── 타겟 클래스 ───────────────────────────────────────────
TARGET_X = {
    1900, 2483, 3351, 3483, 3544, 3743, 3832, 4543,
    12081, 12247, 12778, 13395, 13900, 16232, 16262, 16548,
    16551, 16688, 18147, 18357, 19232, 19552, 19607, 19861,
    20014, 20238, 20877, 21325, 21771, 22074, 22347, 22362,
    24850, 25367, 25438, 25469, 27733, 27777, 27926, 27993,
    28763, 29345, 29451, 29667, 30308, 31863, 31885, 32310,
    33009, 33208, 33880, 34597, 35206, 36637, 38162, 41768,
}
TARGET_N = {
    27653, 23223, 6563, 6192, 31705, 18110, 10221, 21026,
    5094, 44199, 23203, 10224, 33878, 22627, 29871,
    4378, 5886, 12420,
}
TARGET_74 = TARGET_X | TARGET_N



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ── 유틸 함수 ─────────────────────────────────────────────
def extract_group_key(filename):
    m = re.match(r'(K-[\d-]+?)_0_2', os.path.basename(filename))
    return m.group(1) if m else None

def extract_kcode(folder_name):
    m = re.search(r'K-0*(\d+)', folder_name)
    return int(m.group(1)) if m else None

def sample_bg_color(img_array, size=CENTER_SIZE):
    H, W  = img_array.shape[:2]
    cy, cx= H//2, W//2
    half  = size // 2
    patch = img_array[max(0,cy-half):cy+half, max(0,cx-half):cx+half]
    return tuple(patch.mean(axis=(0,1)).astype(np.uint8))

def mask_bbox(img_array, bbox, bg_color, padding=MASK_PADDING):
    if len(bbox) < 4:
        return img_array
    x, y, w, h = [int(v) for v in bbox]
    H, W = img_array.shape[:2]
    x1 = max(0, x-padding);   y1 = max(0, y-padding)
    x2 = min(W, x+w+padding); y2 = min(H, y+h+padding)
    img_array[y1:y2, x1:x2] = bg_color
    return img_array

def build_coco_categories(target_set):
    return [{"id": k, "name": str(k), "supercategory": "pill"}
            for k in sorted(target_set)]



In [ ]:
# ── Step 1: TL 파싱 → 그룹별 수집 ───────────────────────
print("=" * 55)
print("Step 1. TL 파싱")
print("=" * 55)

all_groups = defaultdict(dict)  # group_key → {fn: [(kcode, bbox)]}

for num in ZIP_NUMS:
    tl_path = os.path.join(TEAM_DIR, f"TL_{num}_조합.zip")
    if not os.path.exists(tl_path):
        print(f"  [SKIP] TL_{num} 없음")
        continue
    print(f"  TL_{num}_조합.zip 파싱 중...")
    with zipfile.ZipFile(tl_path, 'r') as z:
        json_files = [f for f in z.namelist() if f.endswith('.json')]
        for jf in json_files:
            try:
                data      = json.loads(z.read(jf))
                img       = data["images"][0]
                ann       = data["annotations"][0]
                fn        = img["file_name"]
                folder    = jf.split('/')[1]
                kcode     = extract_kcode(folder)
                if kcode is None: continue
                if len(ann.get("bbox", [])) < 4: continue
                group_key = extract_group_key(fn)
                if group_key is None: continue
                if fn not in all_groups[group_key]:
                    all_groups[group_key][fn] = []
                all_groups[group_key][fn].append((kcode, ann["bbox"]))
            except: pass

print(f"전체 그룹 수: {len(all_groups)}")



In [ ]:
# ── Step 2: 그룹별 랜덤 1장 선택 ─────────────────────────
print("\n" + "=" * 55)
print("Step 2. 그룹별 랜덤 1장 선택")
print("=" * 55)

selected = {}
for group_key, fn_dict in all_groups.items():
    chosen_fn = random.choice(list(fn_dict.keys()))
    selected[chosen_fn] = fn_dict[chosen_fn]

print(f"선택 후 이미지 수: {len(selected)}장")

# ── Step 3: 필터링 ────────────────────────────────────────
print("\n" + "=" * 55)
print("Step 3. 필터링")
print("=" * 55)

filtered_74 = {
    fn: pills for fn, pills in selected.items()
    if any(k in TARGET_74 for k, _ in pills)
}

print(f"74종 포함 이미지: {len(filtered_74)}장")



In [ ]:
# ── Step 4: TS 마스킹 + 저장 (74종) ─────────────────────────────
print("\n" + "=" * 55)
print("Step 4. 마스킹 + 저장")
print("=" * 55)

coco_74 = {
    "images": [],
    "annotations": [],
    "categories": build_coco_categories(TARGET_74)
}

img_id_74 = ann_id_74 = 1
saved_74 = 0
viz_74 = []

for num in ZIP_NUMS:
    ts_path = os.path.join(TEAM_DIR, f"TS_{num}_조합.zip")
    if not os.path.exists(ts_path):
        print(f"  [SKIP] TS_{num} 없음")
        continue
    print(f"  TS_{num}_조합.zip 처리 중...")

    with zipfile.ZipFile(ts_path, 'r') as z:
        img_map = {
            os.path.basename(f): f
            for f in z.namelist()
            if f.lower().endswith('.png') and 'index' not in f
        }

        # ── 74종 처리 ─────────────────────────────────────
        for fn, pills in filtered_74.items():
            img_path = img_map.get(fn)
            if not img_path:
                continue

            img_bytes = z.read(img_path)
            img_pil = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            img_arr = np.array(img_pil)
            H, W = img_arr.shape[:2]
            bg_color = sample_bg_color(img_arr)

            # TARGET_74에 없는 약은 마스킹
            for kcode, bbox in pills:
                if kcode not in TARGET_74:
                    img_arr = mask_bbox(img_arr, bbox, bg_color)

            Image.fromarray(img_arr).save(os.path.join(OUTPUT_IMG_74, fn))

            coco_74["images"].append({
                "id": img_id_74,
                "file_name": fn,
                "width": W,
                "height": H
            })

            # TARGET_74에 포함된 약만 annotation 저장
            for kcode, bbox in pills:
                if kcode in TARGET_74:
                    coco_74["annotations"].append({
                        "id": ann_id_74,
                        "image_id": img_id_74,
                        "category_id": kcode,
                        "bbox": [float(v) for v in bbox],
                        "area": float(bbox[2] * bbox[3]),
                        "iscrowd": 0,
                    })
                    ann_id_74 += 1

            img_id_74 += 1
            saved_74 += 1

            if len(viz_74) < 10:
                viz_74.append((img_arr.copy(), pills, fn, TARGET_74))

print(f"74종 저장: {saved_74}장 / {ann_id_74 - 1}개 bbox")

In [ ]:
# ── COCO JSON 저장 ────────────────────────────────────────
for coco, ann_dir in [(coco_74, OUTPUT_ANN_74)]:
    p = os.path.join(ann_dir, "_annotations.coco.json")
    with open(p, "w") as f:
        json.dump(coco, f, ensure_ascii=False)
print("COCO JSON 저장 완료")



In [ ]:
# annotation에 있는 category_id만 추출
category_ids = [ann["category_id"] for ann in coco["annotations"]]

unique_ids = sorted(set(category_ids))

print(f"고유 category_id 개수: {len(unique_ids)}")
print("category_id 목록:")
print(unique_ids)

# 74종 중 빠진 category_id or 예상 외 category_id 확인
target_74 = TARGET_74

json_ids = {
    ann["category_id"]
    for ann in coco["annotations"]
}

missing = sorted(target_74 - json_ids)
extra = sorted(json_ids - target_74)

print("빠진 category_id:", missing)
print("예상 외 category_id:", extra)

In [ ]:
# category별 개수 확인
from collections import Counter

# category_id별 annotation 개수
counter = Counter(ann["category_id"] for ann in coco["annotations"])

print(f"전체 category 개수: {len(counter)}\n")

for cid in sorted(counter):
    print(f"{cid}: {counter[cid]}")

### 카테고리별 검수 시각화

In [ ]:
import os
import json
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

# =====================================================
# 설정 (한 번만 수정)
# =====================================================

JSON_PATH = r"/content/drive/MyDrive/_annotations.coco.json"
IMAGE_DIR = r"/content/drive/MyDrive/images"

PADDING = 40
COLS = 5

# =====================================================
# JSON 읽기 (한 번만)
# =====================================================

with open(JSON_PATH, "r", encoding="utf-8") as f:
    coco = json.load(f)

image_map = {img["id"]: img for img in coco["images"]}
category_list = sorted(TARGET_74)


# =====================================================
# 시각화 함수
# =====================================================

def visualize_category(class_index):

    if class_index < 0 or class_index >= len(category_list):
        print(f"class_index는 0 ~ {len(category_list)-1} 사이여야 합니다.")
        return

    target_category = category_list[class_index]

    anns = [
        ann
        for ann in coco["annotations"]
        if ann["category_id"] == target_category
    ]

    print("=" * 60)
    print(f"Class Index : {class_index}")
    print(f"Category ID : {target_category}")
    print(f"Total BBox  : {len(anns)}")
    print("=" * 60)

    if len(anns) == 0:
        print("해당 클래스의 bbox가 없습니다.")
        return

    rows = math.ceil(len(anns) / COLS)

    fig, axes = plt.subplots(
        rows,
        COLS,
        figsize=(4 * COLS, 4 * rows)
    )

    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, ann in zip(axes, anns):

        img_info = image_map[ann["image_id"]]

        img = np.array(
            Image.open(
                os.path.join(IMAGE_DIR, img_info["file_name"])
            ).convert("RGB")
        )

        H, W = img.shape[:2]

        x, y, w, h = map(int, ann["bbox"])

        xmin = max(0, x - PADDING)
        ymin = max(0, y - PADDING)
        xmax = min(W, x + w + PADDING)
        ymax = min(H, y + h + PADDING)

        crop = img[ymin:ymax, xmin:xmax]

        ax.imshow(crop)

        rect = Rectangle(
            (x - xmin, y - ymin),
            w,
            h,
            edgecolor="red",
            facecolor="none",
            linewidth=2,
        )

        ax.add_patch(rect)

        ax.set_title(
            f"ann:{ann['id']}\nimg:{ann['image_id']}",
            fontsize=14
        )

        ax.axis("off")

    plt.tight_layout()
    plt.show()